In [ ]:
import pandas as pd
import csv
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from utils.utils import load_encrypted_xlsx, safe_conversion_to_datetime, ensure_dir
from tqdm import tqdm
import scipy.stats as stats
import itertools

In [ ]:
# registry_data_path="/Users/jonathanheinimann/Documents/University/Dissertation/statistics/for_jh/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx"
# bd_df_path="/Users/jonathanheinimann/Documents/University/Dissertation/statistics/for_jh/20240116_SAH_SOS_Blutdruecke.csv"
# pdms_path="/Users/jonathanheinimann/Documents/University/Dissertation/statistics/for_jh/registry_pdms_correspondence.csv"
# outcome_path="/Users/jonathanheinimann/Documents/University/Dissertation/statistics/for_jh/aSAH_DATA_2009_2024_18122024.xlsx"

registry_data_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/post_hoc_modified_aSAH_DATA_2009_2023_24122023.xlsx'
bd_df_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/extracted_data/20240116_SAH_SOS_Blutdruecke.csv'
pdms_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/pdms_data/registry_pdms_correspondence.csv'
outcome_path = '/Users/jk1/Library/CloudStorage/OneDrive-unige.ch/icu_research/dci_sah/data/sos_sah_data/follow_up/aSAH_DATA_2009_2024_18122024.xlsx'

In [ ]:
registry_df = load_encrypted_xlsx(registry_data_path)
outcome_df = load_encrypted_xlsx(outcome_path)
bp_df = pd.read_csv(bd_df_path, sep= ';', decimal='.')
registry_pdms_correspondance_df = pd.read_csv(pdms_path)

In [ ]:
registry_pdms_correspondance_df.head()

In [ ]:
registry_df.head()

In [ ]:
bp_df.head()

In [ ]:
outcome_df["mRS_FU_1y"]=pd.to_numeric(outcome_df['mRS_FU_1y'], errors='coerce')

In [ ]:
outcome_df["mRS_FU_1y"].unique()

In [ ]:
bp_df=bp_df.merge(registry_pdms_correspondance_df, how='left', on='pNr')

In [ ]:
bp_df.Date_birth.values

In [ ]:
bp_df['Date_birth']=pd.to_datetime(bp_df['Date_birth'], format='%d.%m.%Y')
outcome_df['Date_birth']=pd.to_datetime(outcome_df['Date_birth'])

In [ ]:
print(outcome_df.columns)

In [ ]:
for pnr in tqdm(bp_df["pNr"].unique()):
    sos_center_nr = bp_df[bp_df["pNr"] == pnr]["SOS-CENTER-YEAR-NO."].values[0]
    name = bp_df[bp_df["pNr"] == pnr]["JoinedName"].values[0]
    date_birth = bp_df[bp_df["pNr"] == pnr]["Date_birth"].values[0]
    mrs_values = outcome_df[(outcome_df["SOS-CENTER-YEAR-NO."] == sos_center_nr) &
                        (outcome_df["Name"] == name) &
                        (outcome_df["Date_birth"] == date_birth)]["mRS_FU_1y"]
    if len(mrs_values) == 0:
        mrs = np.nan
    else:
        mrs = mrs_values.values[0]

    bp_df.loc[bp_df["pNr"] == pnr, "mrs_1y"] = mrs



In [ ]:
bp_df.pNr.nunique()

In [ ]:
bp_df.info()

In [ ]:
# TODO drop duplicates in bp_df
bp_df.duplicated(subset=['pNr', 
                         'systole',
                        'diastole',
                        'mitteldruck',
                        'timeBd']).sum()

In [ ]:
# TODO add this to avoid duplicates in BP DF after merge
registry_df.drop_duplicates(inplace=True)
registry_df.dropna(subset=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], inplace=True)

In [ ]:
main_df = bp_df.merge(registry_df, 
                       left_on=['SOS-CENTER-YEAR-NO.', 'JoinedName', 'Date_birth'], 
                       right_on=['SOS-CENTER-YEAR-NO.', 'Name', 'Date_birth'], 
                       how='left')

In [ ]:
# Example of replacing 'none' and NaN with NaT (if needed)
main_df['Date_DCI_ischemia_first_image'] = pd.to_datetime(main_df['Date_DCI_ischemia_first_image'], errors='coerce', format='%Y-%m-%d')
main_df['Time_DCI_ischemia_first_image'] =pd.to_datetime(main_df['Time_DCI_ischemia_first_image'], errors='coerce', format='%H:%M:%S')

main_df['Date_DCI_infarct_first_image'] = pd.to_datetime(main_df['Date_DCI_infarct_first_image'], errors='coerce', format='%Y-%m-%d')
main_df['Date_DCI_infarct_first_image'] = pd. to_datetime(main_df['Date_DCI_infarct_first_image'],errors='coerce', format='%H:%M:%S')
#registry_df['Time_DCI_ischemia_first_image'] = registry_df['Time_DCI_ischemia_first_image'].replace('none', pd.NaT)

In [ ]:
main_df['timestamp_ischemia'] = pd.to_datetime(
    main_df['Date_DCI_ischemia_first_image'].astype(str) + ' ' + main_df['Time_DCI_ischemia_first_image'].astype(str),
    errors='coerce'
)

main_df['timestamp_infarction'] =  pd.to_datetime(
    main_df['Date_DCI_infarct_first_image'].astype(str) + ' ' + main_df['Time_DCI_infarct_first_image'].astype(str),
    errors='coerce'
)

In [ ]:
main_df.head()

In [ ]:
#change timeBd to datetime format
main_df['timeBd']=pd.to_datetime(main_df['timeBd'], format='%Y-%m-%d %H:%M:%S.%f')

In [ ]:
main_df['timeBd'] = main_df['timeBd'].dt.tz_localize(None)
main_df['timestamp_ischemia'] = main_df['timestamp_ischemia'].dt.tz_localize(None)

In [ ]:
main_df['time_difference_ischemia']=main_df['timestamp_ischemia'] - main_df['timeBd']
main_df['time_difference_ischemia']=main_df['time_difference_ischemia'].dt.total_seconds() / 60

main_df['time_difference_infarction']=main_df['timestamp_infarction']-main_df['timeBd']
main_df['time_difference_infarction']=main_df['time_difference_infarction'].dt.total_seconds() / 60

In [ ]:
main_df = main_df.sort_values(by=['pNr', 'timeBd'], ascending=True)

In [ ]:

main_df['T0'] = main_df.groupby('pNr')['timeBd'].transform('min')
main_df['time_difference_to_T0'] = main_df['timeBd'] - main_df['T0']
main_df['time_difference_to_T0'] = main_df['time_difference_to_T0'].dt.total_seconds() / 60

In [ ]:
main_df['time_difference_to_T0'] = pd.to_numeric(main_df['time_difference_to_T0'], errors='coerce')

In [ ]:
auc_df=main_df.copy()

In [ ]:
auc_df['delta_time'] = auc_df['timeBd'].shift(-1) - auc_df['timeBd']
auc_df['delta_time'] = auc_df['delta_time'].dt.total_seconds() / 60

In [ ]:
def product(auc_df):
       # Create a copy to avoid modifying the original dataframe
    result_df = auc_df.copy()
    
    # Calculate product
    result_df['product'] = result_df['delta_time'] * auc_df['systole']
    
    return result_df

In [ ]:
result_df=product(auc_df)

In [ ]:
result_df=result_df[['time_difference_to_T0', 'pNr', 'delta_time', 'systole', 'product', 'DCI_YN_verified', 'mrs_1y']]

In [ ]:
result_df.head()

In [ ]:
# TODO - this thresholds time from initial event / however we would like to threshold the length of the event
def time(tmax):
    global result_df
    time_df=result_df.copy()
    time_df=time_df[time_df['time_difference_to_T0']<= tmax]
    return time_df

In [ ]:
def time_and_hypertension(tmax, threshold):
    result_df=time(tmax)
    result_df['hypertension']=(result_df['systole'] >= threshold).astype(int)
    result_df=result_df[result_df['hypertension']== 1]
    return result_df

In [ ]:
time_and_hypertension(200,200)

In [ ]:
result_df.head()

In [ ]:
len(result_df), result_df.shape

In [ ]:
def threshold_by_insult_intensity(df, intensity_threshold, parameter_name='systole'):
    """
    Define events by as all values above a given intensity threshold, for a given parameter
    Duration of events is recorded as the sum of duration of all consecutive measures
    Product is the sum of all products (intensity * duration) of all consecutive measures

    Arguments:
        - df: pandas Dataframe with all measures (expected columns of timing, duration and product)
        - intensity_threshold: threshold of insult intensity from which the events are defined
        - parameter_name: name of column with measured values in df

    Returns: events_df
    """


    # sum duration of subsequent rows above the intensity threshold
    df['exceeds_intensity_treshold'] = df[parameter_name] >= intensity_threshold
    
    # record all events of intensity > threshold and cumulative duration > threshold
    events_df = pd.DataFrame()
    # loop through all patients
    for pnr in tqdm(df['pNr'].unique()):
        pnr_df = df[df['pNr'] == pnr]
        # exclude values with negative delta_time (last time point of a patient)
        pnr_df = pnr_df[pnr_df['delta_time'] >= 0]

        pnr_df.sort_values(by='time_difference_to_T0', inplace=True)
        
        # event starts where mask is True **and** the previous row was False (or N/A for first row)
        event_starts = pnr_df['exceeds_intensity_treshold'] & ~pnr_df['exceeds_intensity_treshold'].shift(fill_value=False)

        # Cumulative sum of event_starts gives a unique event number; set to 0 where mask is False
        pnr_df["event_id"] = event_starts.cumsum().where(pnr_df['exceeds_intensity_treshold'], 0).astype(int)
        pnr_events_df = pnr_df.groupby('event_id').agg({
                'pNr': 'first',
                'time_difference_to_T0': ['min', 'max'],
                'delta_time': 'sum',
                'product': 'sum',
                'DCI_YN_verified': 'first',
                'mrs_1y': 'first',
            }).reset_index()
        pnr_events_df.columns = ['event_id', 'pNr', 'event_first_measure_rel_time', 'event_last_measure_rel_time', 'event_duration', 'event_product',
                                    'DCI_YN_verified', 'mrs_1y']
        pnr_events_df['intensity_threshold'] = intensity_threshold
        pnr_events_df['parameter_name'] = parameter_name
        # drop the events with id 0
        pnr_events_df = pnr_events_df[pnr_events_df['event_id'] > 0]

        events_df = pd.concat([events_df, pnr_events_df], ignore_index=True)

        # EXPLICIT version
        # # iterate through the rows of the patient's dataframe
        # event_duration = 0
        # event_product = 0
        # event_abs_start_time = None
        # event_rel_start_time = None
        # for i in range(len(pnr_df)):
        #     if pnr_df.iloc[i]['exceeds_intensity_treshold']:
        #         event_duration += pnr_df.iloc[i]['delta_time']
        #         event_product += pnr_df.iloc[i]['product']
        #         if event_abs_start_time is None:
        #             event_rel_start_time = pnr_df.iloc[i]['time_difference_to_T0']

        #         # if next row is not exceeding the threshold, save the event
        #         if i == len(pnr_df) - 1 or not pnr_df.iloc[i + 1]['exceeds_intensity_treshold']:
        #             # define event end (if not last event, this is the date time of the next measure)
        #             if i == len(pnr_df) - 1:
        #                 event_rel_end = pnr_df.iloc[i]['time_difference_to_T0']
        #             else:
        #                 event_rel_end = pnr_df.iloc[i+1]['time_difference_to_T0']

        #             events_df = pd.concat([events_df, pd.DataFrame({
        #                 'pNr': [pnr],
        #                 'event_rel_start_time': [event_rel_start_time],
        #                 'event_rel_end_time': [event_rel_end],
        #                 'event_duration': [event_duration],
        #                 'event_product': [event_product],
        #                 parameter_name: [pnr_df.iloc[i][parameter_name]],
        #                 'intensity_threshold': [intensity_threshold]
        #             })], ignore_index=True)

        #             # reset the event variables
        #             event_duration = 0
        #             event_product = 0
        #             event_abs_start_time, event_rel_start_time = None, None


    return events_df
            

In [ ]:
pnr_df = result_df[result_df.pNr == 13474]
pnr_df['exceeds_intensity_treshold'] = pnr_df['systole'] >= 150
pnr_df.sort_values(by='time_difference_to_T0', inplace=True)
# event starts where mask is True **and** the previous row was False (or N/A for first row)
event_starts = pnr_df['exceeds_intensity_treshold'] & ~pnr_df['exceeds_intensity_treshold'].shift(fill_value=False)

# Cumulative sum of event_starts gives a unique event number; set to 0 where mask is False
pnr_df["event_id"] = event_starts.cumsum().where(pnr_df['exceeds_intensity_treshold'], 0).astype(int)



In [ ]:
pnr_df

In [ ]:
events_df = threshold_by_insult_intensity(result_df, 150, 'systole')

In [ ]:
events_df

In [ ]:
def sum(tmax, threshold):
    sum_df=time_and_hypertension(tmax, threshold)
    sum_df=sum_df.groupby('pNr').agg({
       'product': 'sum',
        'DCI_YN_verified' : 'first',
        'mrs_1y' : 'first'

    }).reset_index()
    return sum_df

In [ ]:
def summarize_all(tmax_range, threshold_range):
    # 1. Basis-Infos einmalig vorbereiten
    base_info = result_df.groupby('pNr').agg({
        'DCI_YN_verified': 'first',
        'mrs_1y': 'first'
        # ggf. weitere Spalten hier
    }).reset_index()

    final_df = None

    # 2. Doppelschleife über tmax und threshold
    for tmax in tmax_range:
        for threshold in threshold_range:
            temp_df = time_and_hypertension(tmax, threshold)
            grouped = temp_df.groupby('pNr').agg({
                'product': 'sum'
            }).reset_index()

            product_column_name = f'product_tmax_{tmax}_threshold_{threshold}'
            grouped = grouped.rename(columns={'product': product_column_name})

            # Merge: initialisieren oder hinzufügen
            if final_df is None:
                final_df = grouped
            else:
                final_df = pd.merge(final_df, grouped, on='pNr', how='outer')

    # 3. Am Ende: Merge mit Basisinfos
    final_df = pd.merge(final_df, base_info, on='pNr', how='left')

    return final_df

In [ ]:
tmax_range = range(1, 1440, 60)
threshold_range = range(100, 200, 5)

result = summarize_all(tmax_range, threshold_range)

In [ ]:
def calculate_by_row(tmax_range, threshold_range):
    # 1. Einmalige Zusatzinformationen
    base_info = result_df.groupby('pNr').agg({
        'DCI_YN_verified': 'first',
        'mrs_1y': 'first'
    }).reset_index()

    all_rows = []

    # 2. Schleife über alle Kombinationen mit Fortschrittsbalken
    for tmax, threshold in tqdm(itertools.product(tmax_range, threshold_range), 
                                total=len(tmax_range)*len(threshold_range), 
                                desc="Berechne Kombinationen"):
        temp_df = time_and_hypertension(tmax, threshold)
        
        # Gruppieren und Summe berechnen
        grouped = temp_df.groupby('pNr').agg({
            'product': 'sum'
        }).reset_index()

        # tmax und threshold als Spalten hinzufügen
        grouped['tmax'] = tmax
        grouped['threshold'] = threshold

        all_rows.append(grouped)

    # 3. Alles zu einem DataFrame zusammenführen
    final_df = pd.concat(all_rows, ignore_index=True)

    # 4. Zusatzinfos mergen
    final_df = pd.merge(final_df, base_info, on='pNr', how='left')

    return final_df

In [ ]:
tmax_range = range(0, 1440, 30)
threshold_range = range(100, 200, 5)

result = calculate_by_row(tmax_range, threshold_range)

In [ ]:
result

In [ ]:
result[result['pNr']==13474]

In [ ]:
print(result)

In [ ]:
def hypertension(threshold):
    global result_df
    result_df=result_df.copy()
    result_df['hypertension']=(result_df['systole'] >= threshold).astype(int)
    result_df=result_df[result_df['hypertension']== 1]
    return result_df

In [ ]:
def sum(threshold):
    sum_df=hypertension(threshold)
    sum_df=sum_df.groupby('pNr')['product'].sum().reset_index()
    sum_df= pd.merge(sum_df, auc_df[['pNr', 'mrs_1y','DCI_YN_verified']], on='pNr')
    sum_df = sum_df.drop_duplicates(subset=['pNr'])
    return sum_df

In [ ]:
sum(130)